In [1]:
import os
import glob
import csv
import warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from astropy.modeling.models import Sersic2D
from astropy.convolution import convolve, Gaussian2DKernel
from astropy.stats import sigma_clipped_stats, SigmaClip
from astropy.visualization import simple_norm
from photutils.segmentation import detect_threshold, detect_sources, SourceCatalog
from astropy.table import Table
import statmorph
from statmorph.utils.image_diagnostics import make_figure
%matplotlib widget

from astropy.io import fits
from photutils.background import Background2D, MedianBackground

from astropy.cosmology import Planck18 as cosmo
from astropy import units as u

from skimage import measure
from skimage import io, color
import skimage.morphology as morph
from skimage.measure import label
from skimage.filters import threshold_otsu

from IPython.display import display
import pandas as pd

In [2]:
def paramFinder(fits_filename=f"fits/temp/tempfits.fits", realpsf=None, segmap=None, mask=None, weightmap=None, verb=False):
    with fits.open(fits_filename) as hdul:
        if verb:
            print(hdul.info())
            print('----------------------------')
            print(hdul[0].header)
            print('----------------------------')
        dims = list(np.shape(hdul[0]))
        image = np.array(hdul[0].data)
        with fits.open(realpsf) as hdul:
            prealpsf = np.array(hdul[0].data)
        with fits.open(segmap) as hdul:
            psegmap = np.array(hdul[0].data)
        with fits.open(mask) as hdul:
            pmask = np.array(hdul[0].data)
        with fits.open(weightmap) as hdul:
            pweightmap = np.array(hdul[0].data)
        
        return image, prealpsf, psegmap, pmask, pweightmap

In [3]:
def subtract_background(image, box_size=50, filter_size=3, sigma=3.0, show=False):
    """
    Subtract background from a simulated galaxy image using Background2D.

    Parameters
    ----------
    image : 2D numpy.ndarray
        The input galaxy image.
    box_size : int or tuple of int, optional
        Size of the box to estimate the background.
    filter_size : int or tuple of int, optional
        Size of the filter to smooth the background map.
    sigma : float, optional
        Sigma level for sigma clipping.
    show : bool, optional
        If True, display the original, background, and subtracted images.

    Returns
    -------
    image_subtracted : 2D numpy.ndarray
        The background-subtracted image.
    background : 2D numpy.ndarray
        The estimated background.
    """
    sigma_clip = SigmaClip(sigma=sigma)
    bkg_estimator = MedianBackground()
    
    bkg = Background2D(image, box_size, filter_size=filter_size,
                       sigma_clip=sigma_clip, bkg_estimator=bkg_estimator)
    
    image_subtracted = image - bkg.background

    if show:
        norm = simple_norm(image, 'sqrt', percent=99)
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(image, origin='lower', norm=norm)
        axs[0].set_title('Original Image')
        axs[1].imshow(bkg.background, origin='lower', norm=norm)
        axs[1].set_title('Estimated Background')
        axs[2].imshow(image_subtracted, origin='lower', norm=norm)
        axs[2].set_title('Background Subtracted')
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return image_subtracted, bkg.background

In [4]:
def get_center_pixel_value(arr):
    """Return the pixel value at the center of a 2D array."""
    center_y = arr.shape[0] // 2
    center_x = arr.shape[1] // 2
    return arr[center_y, center_x]

def find_array_with_largest_center_flux(arrays):
    max_flux = -np.inf
    max_array = None
    max_index = -1

    for i, arr in enumerate(arrays):
        if arr is None:
            continue  # skip None entries
        center_flux = get_center_pixel_value(arr)
        if center_flux > max_flux:
            max_flux = center_flux
            max_array = arr
            max_index = i

    return max_array, max_flux, max_index

In [5]:
def makeSegmap(image):
    Thresh = threshold_otsu(image)
    picBW = image > Thresh
    
    Strel = morph.disk(2)
    BWimg_dil = morph.dilation(picBW,Strel)
    Strel2 = morph.disk(5)
    BWimg_close = morph.closing(BWimg_dil,Strel2)
    
    L = label(BWimg_close)
    half_length = int(np.floor(np.size(L,1)/2))
    L_cntr = L[half_length,half_length]
    Seg = L
    Seg[Seg != L_cntr] = 0
    Seg[Seg == L_cntr] = 1
    Mask = morph.dilation(Seg,Strel2)
    
    #plt.figure(figsize=(7,7))
    #io.imshow(L)
    #plt.show()
    
    return Mask

In [6]:
def runStatMorph(image, psf, segmap, mask, weightmap, gain=None, center_coords=None):
    """
    Run StatMorph on the central galaxy.
    If the model fails (None), automatically try the next closest galaxy until a valid Sérsic model is found.

    Parameters
    ----------
    center_coords : tuple, optional
        (x_pix, y_pix) coordinates of the target galaxy center.
        If None, defaults to image center.
    """
    import numpy as np
    import statmorph
    from astropy.convolution import Gaussian2DKernel
    from scipy.spatial import distance

    mask = mask.astype(bool)

    # --- Use Gaussian PSF if none provided ---
    if psf is None:
        kernel = Gaussian2DKernel(1.5)
        kernel.normalize()
        psf = kernel.array

    # --- Determine center ---
    H, W = image.shape
    center = np.array(center_coords) if center_coords is not None else np.array([W/2, H/2])

    # --- Identify galaxies ---
    labels = np.unique(segmap)
    labels = labels[labels > 0]  # ignore background
    if len(labels) == 0:
        return mask, [], []

    # --- Sort galaxies by distance to center ---
    dist_list = []
    for lbl in labels:
        yx = np.argwhere(segmap == lbl)
        yc, xc = yx.mean(axis=0)
        dist_list.append((lbl, distance.euclidean([xc, yc], center)))
    dist_list.sort(key=lambda t: t[1])

    # --- Try each galaxy until valid model found ---
    for lbl, _ in dist_list:
        segmap_tmp = np.where(segmap == lbl, segmap, 0)
        try:
            morph_list = statmorph.source_morphology(
                image, segmap_tmp, weightmap=weightmap, gain=gain, psf=psf, mask=mask
            )
        except Exception:
            continue

        ny, nx = image.shape
        y, x = np.mgrid[0:ny, 0:nx]
        image_model = []
        for morph in morph_list:
            if morph.flag_sersic != 4:
                model = statmorph.ConvolvedSersic2D(
                    morph.sersic_amplitude,
                    morph.sersic_rhalf,
                    morph.sersic_n,
                    morph.sersic_xc,
                    morph.sersic_yc,
                    morph.sersic_ellip,
                    morph.sersic_theta
                )
                model.set_psf(psf)
                image_model.append(model(x, y))
            else:
                image_model.append(None)

        for im in image_model:
            if im is not None:
                return mask, morph_list, image_model

    return mask, [], []

def formatStatMorph(morph, verb=False):
    lines = []

    if verb:
        lines.append('BASIC MEASUREMENTS (NON-PARAMETRIC)')
        lines.append(f'xc_centroid = {morph.xc_centroid}')
        lines.append(f'yc_centroid = {morph.yc_centroid}')
        lines.append(f'ellipticity_centroid = {morph.ellipticity_centroid}')
        lines.append(f'elongation_centroid = {morph.elongation_centroid}')
        lines.append(f'orientation_centroid = {morph.orientation_centroid}')
        lines.append(f'xc_asymmetry = {morph.xc_asymmetry}')
        lines.append(f'yc_asymmetry = {morph.yc_asymmetry}')
        lines.append(f'ellipticity_asymmetry = {morph.ellipticity_asymmetry}')
        lines.append(f'elongation_asymmetry = {morph.elongation_asymmetry}')
        lines.append(f'orientation_asymmetry = {morph.orientation_asymmetry}')
        lines.append(f'rpetro_circ = {morph.rpetro_circ}')
        lines.append(f'rpetro_ellip = {morph.rpetro_ellip}')
        lines.append(f'rhalf_circ = {morph.rhalf_circ}')
        lines.append(f'rhalf_ellip = {morph.rhalf_ellip}')
        lines.append(f'r20 = {morph.r20}')
        lines.append(f'r80 = {morph.r80}')
        lines.append(f'Gini = {morph.gini}')
        lines.append(f'M20 = {morph.m20}')
        lines.append(f'F(G, M20) = {morph.gini_m20_bulge}')
        lines.append(f'S(G, M20) = {morph.gini_m20_merger}')
        lines.append(f'sn_per_pixel = {morph.sn_per_pixel}')
        lines.append(f'C = {morph.concentration}')
        lines.append(f'A = {morph.asymmetry}')
        lines.append(f'S = {morph.smoothness}')
        lines.append('')

    lines.append('SERSIC MODEL')
    lines.append(f'sersic_amplitude = {morph.sersic_amplitude}')
    lines.append(f'sersic_rhalf = {morph.sersic_rhalf}')
    lines.append(f'sersic_n = {morph.sersic_n}')
    if verb:
        lines.append(f'sersic_xc = {morph.sersic_xc}')
        lines.append(f'sersic_yc = {morph.sersic_yc}')
        lines.append(f'sersic_ellip = {morph.sersic_ellip}')
    lines.append(f'sersic_b/a = {1-morph.sersic_ellip}')
    PA = (90 + np.degrees(morph.sersic_theta) % 180)
    if PA > 90:
        PA = PA - 180
    lines.append(f'sersic_theta = {PA}')
    lines.append(f'sersic_chi2_dof = {morph.sersic_chi2_dof}')
    lines.append(f'sky_mean = {morph.sky_mean}')
    if verb:
        lines.append(f'sky_median = {morph.sky_median}')
        lines.append(f'sky_sigma = {morph.sky_sigma}')
    lines.append(f'flag = {morph.flag}')
    if verb:
        lines.append(f'flag_sersic = {morph.flag_sersic}')

    return "\n".join(lines)

def extractFITSRow(morph, galaxy_id, metadata_csv=None, errors=None):
    """
    Build a single FITS row dictionary from a StatMorph result,
    automatically reading zeropoint and survey mode from the metadata CSV.
    """
    # --- Default values if CSV not provided ---
    zp = 27.17
    survey = "CEERS"

    if metadata_csv is not None:
        with open(metadata_csv, newline="") as f:
            reader = csv.DictReader(f)
            row_meta = next(reader)  # Single-row CSV
            zp = float(row_meta.get("zp", zp))
            survey = row_meta.get("mode", survey).upper()

    row = {}
    row['Id'] = float(galaxy_id)       # unified ID
    row['Survey'] = survey             # CEERS or JADES

    # Circular Petrosian flux → AB magnitude
    try:
        flux = morph.flux_circ
        if flux > 0:
            mag = -2.5 * np.log10(flux) + zp
        else:
            mag = np.nan
    except Exception:
        mag = np.nan
    row['mag'] = mag

    # Sérsic parameters
    row['R_e'] = morph.sersic_rhalf
    row['n'] = morph.sersic_n
    row['q'] = 1.0 - morph.sersic_ellip

    PA = (90 + np.degrees(morph.sersic_theta) % 180)
    if PA > 90:
        PA -= 180
    row['P.A.'] = PA

    # Optional errors
    row['e_mag'] = errors.get('mag', np.nan) if errors else np.nan
    row['e_Re'] = errors.get('R_e', np.nan) if errors else np.nan
    row['e_n'] = errors.get('n', np.nan) if errors else np.nan
    row['e_q'] = errors.get('q', np.nan) if errors else np.nan
    row['e_PA'] = errors.get('P.A.', np.nan) if errors else np.nan
    row['Chi^2/nu'] = morph.sersic_chi2_dof

    return row

def write_statmorph_fits(path, fits_rows):
    """
    Write a list of dictionaries into a FITS table with a defined structure.
    """
    if not fits_rows:
        print("No morphological results to write.")
        return

    # Define exact column order
    columns = [
        'Id', 'Survey', 'mag', 'R_e', 'n', 'q', 'P.A.',
        'e_mag', 'e_Re', 'e_n', 'e_q', 'e_PA', 'Chi^2/nu'
    ]

    table_data = {key: [row.get(key, np.nan) for row in fits_rows] for key in columns}
    table = Table(table_data)

    os.makedirs(os.path.dirname(path), exist_ok=True)
    table.write(path, format='fits', overwrite=True)
    print(f"Written custom statmorph summary to: {path}")


In [7]:
def kpc_to_pixels(kpc, z, arcsec_per_pixel=0.01):
    # Use angular diameter distance instead of comoving distance
    distance_kpc = cosmo.angular_diameter_distance(z).to(u.kpc).value
    angular_size_arcsec = (kpc / distance_kpc) * 206265  # arcsec
    length_pixels = angular_size_arcsec / arcsec_per_pixel
    return length_pixels

def pixels_to_kpc(pixels, redshift, pixel_scale=0.01):
    """
    Convert pixels to physical distance (kpc) given redshift.

    Parameters:
    - pixels (float): Number of pixels.
    - redshift (float): Redshift of the object.
    - pixel_scale (float): Arcseconds per pixel (default is 0.01 arcsec/pixel).

    Returns:
    - distance_kpc (float): Physical distance in kiloparsecs.
    """
    # Total angular size in arcsec
    angular_size_arcsec = pixels * pixel_scale * u.arcsec

    # Convert to radians as a plain float
    angular_size_rad = angular_size_arcsec.to(u.rad).value

    # Angular diameter distance in kpc
    ang_diam_dist_kpc = cosmo.angular_diameter_distance(redshift).to(u.kpc).value

    # Physical size = angle (rad) * angular diameter distance (kpc)
    physical_size_kpc = angular_size_rad * ang_diam_dist_kpc

    return physical_size_kpc

#print(pixels_to_kpc(kpc_to_pixels(2, 2), 2))

In [8]:
def plotMorph(morph):
    plt.figure()
    fig = make_figure(morph)
    plt.show()

def plotImage(imageFITS, image_model, export=False, savepath=None):
    if image_model is None:
        print("No valid Sérsic model to plot.")
        return
    
    fig, ax = plt.subplots(1, 3, figsize=(8, 3))
    ax[0].imshow(imageFITS, origin='lower', cmap='viridis',
                 norm=simple_norm(imageFITS, stretch='log', log_a=10000))
    ax[0].set_title("Raw FITS")

    ax[1].imshow(image_model, origin='lower', cmap='viridis',
                 norm=simple_norm(imageFITS, stretch='log', log_a=10000))
    ax[1].set_title("Sérsic Model")

    residual = imageFITS - image_model
    ax[2].imshow(residual, origin='lower', cmap='viridis',
                 norm=simple_norm(residual, stretch='linear'))
    ax[2].set_title("Residual")

    plt.tight_layout()
    if export and savepath:
        plt.savefig(savepath)
    plt.show()


In [9]:
def galaxy_sort(folder_path):
    galaxy_names = set()

    for filename in os.listdir(folder_path):
        # Skip files containing "psf" (case-insensitive)
        if "psf" in filename.lower() or "f150w" in filename.lower():
            continue

        if filename.endswith(".fits") and "_" in filename:
            _, _, galaxy_part = filename.rpartition('_')
            galaxy_name = galaxy_part.rsplit('.fits', 1)[0]
            galaxy_names.add(galaxy_name)

    return sorted(galaxy_names)

In [10]:
def statMain(path_tag, file_tag, filename, realpsf, segmap, mask, weightmap,
             gain=1.0, export=False, rmBG=False, verb=False, verbf=False,
             plotted=[True, True], center_coords=None, n_filter=15, cnt_max=5, crop_margin=20):
    """
    Wrapper for running StatMorph:
    - Handles loading and optional background subtraction
    - Calls runStatMorph which finds the first valid Sérsic model
    - Iteratively crops image if Sérsic index is extreme
    - Returns the *best* (lowest chi²/ν) model even if all others fail
    """

    import numpy as np

    # --- Load inputs ---
    imageFITS, realpsf, segmap, mask, weightmap = paramFinder(
        filename, realpsf, segmap, mask, weightmap
    )

    if rmBG:
        imageFITS, _ = subtract_background(imageFITS)

    # --- Prepare record of all fits ---
    fit_history = []  # (morph, image_model, imageFITS_crop, chi2, crop_iter)

    # --- Initial StatMorph run ---
    segm, morph_list, image_model_list = runStatMorph(
        imageFITS, realpsf, segmap, mask, weightmap, gain, center_coords=center_coords
    )

    # --- Collect valid fits (if any) ---
    for m, im in zip(morph_list, image_model_list):
        if im is not None and hasattr(m, "sersic_chi2_dof"):
            fit_history.append((m, im, imageFITS.copy(), m.sersic_chi2_dof, 0))

    # --- Re-run on cropped image if Sérsic index extreme ---
    cnt = 0

    while cnt < cnt_max:
        if len(fit_history) > 0 and fit_history[-1][0].sersic_n <= n_filter:
            break  # satisfactory n — stop early

        # --- Compute crop margin ---
        H, W = imageFITS.shape
        margin = int(min(H, W) * 0.05) if (fit_history and fit_history[-1][0].sersic_n > n_filter * 2) else crop_margin

        # --- Crop safely ---
        if H - 2 * margin < 50 or W - 2 * margin < 50:
            print("Reached minimum crop size — stopping.")
            break

        imageFITS = imageFITS[margin:-margin, margin:-margin]
        segmap = segmap[margin:-margin, margin:-margin]
        mask = mask[margin:-margin, margin:-margin]
        weightmap = weightmap[margin:-margin, margin:-margin]

        segm, morph_list, image_model_list = runStatMorph(
            imageFITS, realpsf, segmap, mask, weightmap,
            gain, center_coords=center_coords
        )

        valid_models = [(m, im) for m, im in zip(morph_list, image_model_list) if im is not None]
        if len(valid_models) == 0:
            cnt += 1
            if verb:
                print(f"Iteration {cnt}: No valid Sérsic model, continuing.")
            continue

        morph, image_model = valid_models[0]
        if hasattr(morph, "sersic_chi2_dof"):
            fit_history.append((morph, image_model, imageFITS.copy(), morph.sersic_chi2_dof, cnt))

        if verb:
            print(f"Iteration {cnt}: Cropped margin={margin}, Sérsic n={morph.sersic_n:.2f}, chi²/ν={morph.sersic_chi2_dof:.3f}")
        cnt += 1

    # --- Choose the best (lowest chi²/ν) among all valid fits ---
    if len(fit_history) == 0:
        print("Warning: No valid Sérsic models found in any iteration.")
        return None, [None, imageFITS], None

    best_fit = min(fit_history, key=lambda x: x[3])  # (morph, model, image, chi2, iter)
    morph, image_model, image_best, chi2_best, iter_best = best_fit

    if verb:
        print(f"\nSelected best fit from iteration {iter_best} with χ²/ν = {chi2_best:.3f}")

    # --- Verbose formatting ---
    morph_info = formatStatMorph(morph, verbf) if verb else None
    if verb and morph_info:
        print(morph_info, "\n")

    # --- Optional plotting ---
    if plotted[0]:
        plotMorph(morph)
    if plotted[1]:
        plotImage(image_best, image_model, export,
                  f'{path_tag}/results/sersic_morph_galaxy_{file_tag}.png')

    if export:
        return morph, [image_model, image_best], morph_info
    else:
        return morph, [image_model, image_best], None


In [11]:
def mainWrapper(path_tag, centers_csv=None, export=True, verb=True, plotted=[False, True],
                save_model_fits=False):
    import os, glob, csv
    import pandas as pd
    from astropy.io import fits as pyfits

    files = galaxy_sort(path_tag)
    morph_text = []
    fits_rows = []
    failed_galaxies = []  # track failed galaxies

    # --- Load PSF ---
    psf_dir = os.path.join(path_tag, "psf")
    psf_files = glob.glob(os.path.join(psf_dir, "*.fits"))
    if len(psf_files) == 0:
        raise FileNotFoundError(f"No PSF files found in {psf_dir}")
    elif len(psf_files) > 1:
        print(f"Multiple PSF files found, using {psf_files[0]}")
    realpsf = psf_files[0]

    # --- Load metadata CSV ---
    metadata_csv = os.path.join(path_tag, "galaxy_metadata.csv")
    if not os.path.exists(metadata_csv):
        raise FileNotFoundError(f"Metadata CSV not found: {metadata_csv}")

    with open(metadata_csv, newline="") as f:
        reader = csv.DictReader(f)
        meta_row = next(reader)
        zp = float(meta_row.get("zp", 27.17))
        survey = meta_row.get("mode", "CEERS").upper()

    # --- Load galaxy centers CSV if provided ---
    if centers_csv is not None:
        centers_df = pd.read_csv(centers_csv)
        centers_map = {str(int(row.gal_id)): (row.x_pix, row.y_pix) for row in centers_df.itertuples()}
    else:
        centers_map = {}

    # --- Prepare optional fits folder ---
    fits_dir = os.path.join(path_tag, "results", "fits") if save_model_fits else None
    if fits_dir:
        os.makedirs(fits_dir, exist_ok=True)

    # --- Loop through galaxies ---
    for file_tag in files:
        print(f"\n------------------\nGalaxy {file_tag}\n------------------\n")

        filename  = f'{path_tag}/cutout_sci_{file_tag}.fits'
        segmap    = f'{path_tag}/segmap_{file_tag}.fits'
        mask      = f'{path_tag}/mask_{file_tag}.fits'
        weightmap = f'{path_tag}/cutout_err_{file_tag}.fits'

        morph_text.append(file_tag)
        center_coords = centers_map.get(file_tag, None)

        # --- Run StatMorph safely ---
        try:
            morph, [image_model, imageFITS], morph_info = statMain(
                path_tag, file_tag, filename,
                realpsf, segmap, mask, weightmap,
                export=True, verb=verb, plotted=plotted,
                center_coords=center_coords
            )
        except Exception as e:
            print(f"Error processing galaxy {file_tag}: {e}")
            failed_galaxies.append(file_tag)
            morph_text.append("Failed to process this galaxy.\n")
            continue

        # --- Handle cases where StatMorph returns None ---
        if morph is None:
            print(f"No valid Sérsic model for galaxy {file_tag}. Skipping.")
            failed_galaxies.append(file_tag)
            morph_text.append("No valid Sérsic model found.\n")
            continue

        # --- Extract FITS row safely ---
        try:
            galaxy_id = float(file_tag)
            fits_row = extractFITSRow(morph, galaxy_id, metadata_csv=metadata_csv)
            fits_rows.append(fits_row)
        except Exception as e:
            print(f"Could not extract FITS row for galaxy {file_tag}: {e}")
            failed_galaxies.append(file_tag)

        morph_text.append(morph_info or "No morph info available.\n")

        # --- Optionally save model FITS ---
        if save_model_fits:
            model_fits_path = os.path.join(fits_dir, f"sersic_morph_galaxy_{file_tag}_model.fits")
            hdu = pyfits.PrimaryHDU(image_model)
            hdu.writeto(model_fits_path, overwrite=True)
            print(f"Saved model FITS for galaxy {file_tag} to {model_fits_path}")

    # --- Export results ---
    if export:
        results_dir = os.path.join(path_tag, "results")
        os.makedirs(results_dir, exist_ok=True)

        # --- Text summary ---
        text_file = os.path.join(results_dir, "sersic_morph_data.txt")
        with open(text_file, "w") as f:
            for i in range(0, len(morph_text), 2):
                galaxy_name = morph_text[i]
                morph_next = morph_text[i+1] if i+1 < len(morph_text) else "No data."
                f.write(f"{galaxy_name}:\n{morph_next}\n\n")
        print(f"Text summary saved to {text_file}")

        # --- FITS table ---
        if fits_rows:
            fits_file = os.path.join(results_dir, "statmorph_summary.fits")
            write_statmorph_fits(fits_file, fits_rows)
        else:
            print("No valid morphological results — FITS table not written.")

        # --- Failed galaxy log ---
        if failed_galaxies:
            fail_file = os.path.join(results_dir, "failed_galaxies.txt")
            with open(fail_file, "w") as f:
                for g in failed_galaxies:
                    f.write(f"{g}\n")
            print(f"Logged {len(failed_galaxies)} failed galaxies to {fail_file}")

    print("\nProcessing complete.")


if __name__ == "__main__":
    morph_text = []
    plt.close('all')
    warnings.filterwarnings("ignore", module="statmorph")
    path_tag = 'galaxy_cutouts/ceers/f115w'
    mainWrapper(path_tag, export=True, verb=True, plotted=[False, True])
    
    #frame, img, imageFITS = statMain(filename, wave=3, rmBG=False, verb=False, plotted=[True, True])
    

morph_text = []
plt.close('all')
warnings.filterwarnings("ignore", module="statmorph")
path_tag = 'galaxy_cutouts/jades/f115w'
mainWrapper(path_tag, export=True, verb=True, plotted=[False, True])
    
#frame, img, imageFITS = statMain(filename, wave=3, rmBG=False, verb=False, plotted=[True, True])
    

In [14]:
def galLibTestFunc(): #Import check for wrapper functions
    return 0